In [9]:
import pandas as pd
import tensorflow as tf

NUM_OF_TOKENS = 10

In [10]:
NUM_OF_TOKENS = 11

def prepare_data(file_path, num_prev=NUM_OF_TOKENS-1):
    """

    Prepare data for the transformer model.

    Args:
        file_path (str): Path to the CSV file.
        num_prev (int): Number of previous time frames to include.

    Returns:
        input_tensor (tf.Tensor): Tensor with shape (num_samples, num_prev + 1, 4).
        target_tensor (tf.Tensor): Tensor with shape (num_samples,).
    """
    df = pd.read_csv(file_path)

    # Create shifted columns for previous time frames
    for i in range(1, num_prev + 1):
        df[f'open-prev-{i}'] = df['open_normalized'].shift(i)
        df[f'high-prev-{i}'] = df['high_normalized'].shift(i)
        df[f'low-prev-{i}'] = df['low_normalized'].shift(i)
        df[f'close-prev-{i}'] = df['close_normalized'].shift(i)

    # Remove rows with missing values
    df.dropna(inplace=True)

    # Convert target column to tensor
    target_tensor = tf.convert_to_tensor(df['target'].values, dtype=tf.int32)

    # Drop the target column from input features
    df = df.drop(columns=['target'])

    # Convert input features to tensor
    input_tensor = tf.convert_to_tensor(df.values, dtype=tf.float32)

    # Reshape the input tensor to (num_samples, num_prev + 1, 4)
    num_samples = input_tensor.shape[0]
    input_tensor = tf.reshape(input_tensor, (num_samples, num_prev + 1, 4))

    return input_tensor, target_tensor


In [11]:
input_tensor_train, target_tensor_train = prepare_data('training/ADAUSD/train.csv')
input_tensor_val, target_tensor_val = prepare_data('training/ADAUSD/val.csv')
input_tensor_test, target_tensor_test = prepare_data('training/ADAUSD/test.csv')

KeyboardInterrupt: 

In [ ]:
from tensorflow.keras.layers import Input, Dense, LayerNormalization, Dropout
from tensorflow.keras.models import Model
import numpy as np

# Positional Encoding remains the same
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, maxlen, d_model):
        super(PositionalEncoding, self).__init__()
        self.pos_encoding = self.positional_encoding(maxlen, d_model)

    def positional_encoding(self, maxlen, d_model):
        import numpy as np
        positions = np.arange(maxlen)[:, np.newaxis]
        angles = np.arange(d_model)[np.newaxis, :]
        angle_rates = 1 / np.power(10000, (2 * (angles // 2)) / np.float32(d_model))
        angle_rads = positions * angle_rates

        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

        return tf.cast(angle_rads[np.newaxis, ...], dtype=tf.float32)

    def call(self, inputs):
        return inputs + self.pos_encoding[:, :tf.shape(inputs)[1], :]

# Transformer Block remains the same
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [Dense(ff_dim, activation="relu"), Dense(embed_dim)]
        )
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Transformer Model
def build_transformer_model(input_shape, num_classes, d_model=32, num_heads=2, ff_dim=64, num_layers=2):
    inputs = Input(shape=input_shape)
    
    # Project the input (4 features per token) to the embedding dimension (d_model)
    x = Dense(d_model)(inputs)
    
    # Positional Encoding
    x = PositionalEncoding(input_shape[0], d_model)(x)
    
    # Transformer blocks
    for _ in range(num_layers):
        x = TransformerBlock(d_model, num_heads, ff_dim)(x)
    
    x = Dense(128, activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.1)(x)
    
    # Output layer
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

# Rebuild and compile the model
input_shape = (NUM_OF_TOKENS, 4)  # 11 tokens, 4 features per token
num_classes = 3  # 3 possible classes for the target

model = build_transformer_model(input_shape, num_classes, d_model=64, num_layers=5)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


In [ ]:
# Train the model
history = model.fit(input_tensor_train, target_tensor_train, epochs=10, batch_size=32, 
                    validation_data=(input_tensor_val, target_tensor_val))

# Evaluate the model
model.evaluate(input_tensor_test, target_tensor_test)


Epoch 1/10


2024-09-22 17:10:07.014289: I tensorflow/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-09-22 17:10:07.758691: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x55d81d3a2320 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-09-22 17:10:07.758744: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 3070 Ti Laptop GPU, Compute Capability 8.6
2024-09-22 17:10:07.762875: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2024-09-22 17:10:07.775050: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8700
2024-09-22 17:10:07.838632: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


28628/28628 [==============================] - 911s 31ms/step - loss: 0.4045 - accuracy: 0.8261 - val_loss: 0.3154 - val_accuracy: 0.8809
Epoch 2/10
28628/28628 [==============================] - 869s 30ms/step - loss: 0.3809 - accuracy: 0.8261 - val_loss: 0.3102 - val_accuracy: 0.8809
Epoch 3/10
28628/28628 [==============================] - 866s 30ms/step - loss: 0.3859 - accuracy: 0.8261 - val_loss: 0.3663 - val_accuracy: 0.8809
Epoch 4/10
28628/28628 [==============================] - 828s 29ms/step - loss: 0.3843 - accuracy: 0.8261 - val_loss: 0.2877 - val_accuracy: 0.8809
Epoch 5/10
 4071/28628 [===>..........................] - ETA: 11:14 - loss: 0.3743 - accuracy: 0.8243

KeyboardInterrupt: 